# 03 — Model Interrogation

**What this notebook does:** loads the deployable continuity-risk model and scores one or many SIRENs, with a built-in diagnostic to inspect what the data lake actually contains for a given company.

**What it does NOT do:** train the model (see `02_training.ipynb`), refresh the data lake (see `01_data_pipeline.ipynb`).

**How a score is produced.**

1. The model expects one feature vector per call. That vector is read from `data-lake/features/company_year_features/` — the precomputed table built by `01_data_pipeline.ipynb`.
2. For a given SIREN we pick the *most recent* `prediction_year` available (highest cutoff date).
3. The model returns a raw probability. A calibrator (isotonic regression) maps it to a well-calibrated probability if one is present in `ml-artifacts/`.
4. We assign a risk band from the calibrated score (or raw if no calibrator): `low < watch < medium < high`.

**Two crucial properties of the score.**

- **Raw vs calibrated.** The raw HGB score is inflated because training uses `class_weight='balanced'` (necessary because closures are rare). It's useful for *ranking* companies relative to each other but not for absolute probabilities. The calibrated score is the actual probability estimate. Always read the calibrated number when present.
- **The cutoff date is everything.** A feature row with `prediction_year=2024` describes the company as of Dec 31, 2024 — events from 2025 or 2026 are *invisible* to it. If a company collapses after the cutoff, the score will not reflect it. To capture recent events, the data lake must be current AND the features must be rebuilt with a later cutoff (`01_data_pipeline.ipynb`).

## Setup

Same as the data and training notebooks.

In [ ]:
from pathlib import Path
import os, subprocess, sys, json

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'ml-workflow'
REPO_DIR = Path('/content/pfein')
BACKEND_DIR = REPO_DIR / 'back_end'

cwd = Path.cwd()
if (cwd / 'collabs' / 'requirements-colab.txt').exists():
    BACKEND_DIR = cwd
    REPO_DIR = BACKEND_DIR.parent

if not (BACKEND_DIR / 'collabs' / 'requirements-colab.txt').exists():
    if not (REPO_DIR / '.git').exists():
        subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin'])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'switch', BRANCH])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH])

DRIVE_ROOT = Path('/content/drive/MyDrive/PFE ML Data/pfe_data')
DATA_LAKE = DRIVE_ROOT / 'data-lake'
ARTIFACTS_DIR = DRIVE_ROOT / 'ml-artifacts'
DUCKDB_TMP = Path('/content/pfein_duckdb_tmp')
for p in (DATA_LAKE, ARTIFACTS_DIR, DUCKDB_TMP):
    p.mkdir(parents=True, exist_ok=True)
os.environ['DUCKDB_TEMP_DIRECTORY'] = str(DUCKDB_TMP)

os.chdir(BACKEND_DIR)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(BACKEND_DIR / 'collabs' / 'requirements-colab.txt')])

print(f'BACKEND_DIR   = {BACKEND_DIR}')
print(f'DATA_LAKE     = {DATA_LAKE}')
print(f'ARTIFACTS_DIR = {ARTIFACTS_DIR}')

## Pick the model artifact

Most of the time you want `ml-artifacts/model.joblib` (the deployable artifact). If you want to score against a specific archived run, set `SELECTED_RUN_NAME`.

In [ ]:
SELECTED_RUN_NAME = None  # e.g. 'continuity-risk-20260516-001351_continuity_hgb_full_dataset_900000'

if SELECTED_RUN_NAME:
    run_dir = ARTIFACTS_DIR / 'runs' / SELECTED_RUN_NAME
    SELECTED_MODEL_PATH = run_dir / 'model.joblib'
    SELECTED_METADATA_PATH = run_dir / 'metadata.json' if (run_dir / 'metadata.json').exists() else ARTIFACTS_DIR / 'model_metadata.json'
else:
    SELECTED_MODEL_PATH = ARTIFACTS_DIR / 'model.joblib'
    SELECTED_METADATA_PATH = ARTIFACTS_DIR / 'model_metadata.json'

for required in (SELECTED_MODEL_PATH, SELECTED_METADATA_PATH):
    if not required.exists():
        raise FileNotFoundError(f'Missing artifact: {required}. Train one in 02_training.ipynb first.')

selected_metadata = json.loads(SELECTED_METADATA_PATH.read_text(encoding='utf-8'))
print(f'Model artifact:   {SELECTED_MODEL_PATH}')
print(f'Model version:    {selected_metadata.get("model_version")}')
print(f'Run name:         {selected_metadata.get("run_name")}')
print(f'Family:           {selected_metadata.get("model_family")}')
print(f'Train years:      {selected_metadata.get("train_start_year")} - {selected_metadata.get("train_end_year")}')
print(f'Average precision (test): {selected_metadata.get("metrics", {}).get("average_precision")}')
print(f'ROC-AUC (test):           {selected_metadata.get("metrics", {}).get("roc_auc")}')

## Load the model and define scoring helpers

**Three non-obvious things this cell handles.**

1. **Pickle namespace.** The training script ran as `__main__`, so the pickled pipeline references `__main__.CategoricalCardinalityCapper` (and a few other custom classes). To unpickle in this notebook (where `__main__` is the notebook itself), we copy those classes onto `sys.modules['__main__']` before calling `joblib.load`. Without this, you get `AttributeError: Can't get attribute 'CategoricalCardinalityCapper' on <module '__main__'>`.
2. **Bundle unwrap.** `train_continuity_model.py` saves a dict like `{'pipeline': <estimator>, 'feature_columns': [...], 'target': ..., 'horizon_months': 12}`, not the bare estimator. We extract `'pipeline'` so `model.predict_proba(X)` works.
3. **Confidence score.** Alongside `probability_calibrated`, every scored row gets a `confidence` field (0–1). It's `sqrt(decisiveness × data_quality)` where:
   - **decisiveness** = `2 × |probability - 0.5|` (1.0 when the model commits, 0.0 when it sits on the fence)
   - **data_quality** = weighted sum of four signals (identity present, financials present, recent filing, cutoff freshness)
   - **geometric mean** so confidence is low if *either* component is low — a decisive prediction on a company with no financials still flags as low confidence.

   The `confidence_notes` column explains why (e.g. *"low confidence (no financials filed, cutoff 3y old)"*). Tune `CONFIDENCE_WEIGHTS` in the cell below to reflect what your users care about.

In [ ]:
import re
import duckdb
import joblib
import numpy as np
import pandas as pd
from datetime import date
from IPython.display import display

# (1) Re-alias the training module's classes onto __main__ so unpickle can find them.
from app.tools import train_continuity_model as _train_continuity_model  # noqa: F401
_main_module = sys.modules["__main__"]
for _name in dir(_train_continuity_model):
    _obj = getattr(_train_continuity_model, _name)
    if isinstance(_obj, type) and getattr(_obj, "__module__", None) == _train_continuity_model.__name__:
        setattr(_main_module, _name, _obj)

# (2) Load and unwrap the bundle.
_loaded_artifact = joblib.load(SELECTED_MODEL_PATH)
if isinstance(_loaded_artifact, dict) and "pipeline" in _loaded_artifact:
    model = _loaded_artifact["pipeline"]
    bundle_feature_columns = _loaded_artifact.get("feature_columns")
else:
    model = _loaded_artifact
    bundle_feature_columns = None

# Optional probability calibrator (isotonic regression fit on a holdout).
calibrator = None
calibrator_path = None
for candidate in [
    SELECTED_MODEL_PATH.parent / "isotonic.joblib",
    SELECTED_MODEL_PATH.parent / "isotonic_calibrator.joblib",
    SELECTED_MODEL_PATH.parent / "platt.joblib",
    ARTIFACTS_DIR / "isotonic.joblib",
    ARTIFACTS_DIR / "isotonic_calibrator.joblib",
    ARTIFACTS_DIR / "platt.joblib",
]:
    if candidate.exists():
        calibrator_path = candidate
        calibrator = joblib.load(candidate)
        break

if calibrator_path:
    print(f"Loaded probability calibrator: {calibrator_path}")
else:
    print("No probability calibrator found. Raw scores will be used for the risk band — useful for ranking, but probabilities may be inflated.")

# --- Feature dataset path ---
FEATURES_PATH = DATA_LAKE / "features" / "company_year_features"
if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Missing feature dataset: {FEATURES_PATH}. Build it with 01_data_pipeline.ipynb.")
FEATURES_GLOB_SQL = (FEATURES_PATH / "**" / "*.parquet").as_posix().replace("'", "''")

def sql_literal(value):
    s = str(value).replace("'", "''")
    return "'" + s + "'"

def normalize_siren(value):
    digits = re.sub(r"\D", "", str(value))
    if len(digits) != 9:
        raise ValueError(f"SIREN must be 9 digits, got {value!r}")
    return digits

def expected_feature_columns():
    if bundle_feature_columns:
        return list(bundle_feature_columns)
    columns = selected_metadata.get("feature_columns")
    if columns:
        return list(columns)
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)
    if hasattr(model, "named_steps"):
        for step in model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)
    raise RuntimeError("Could not determine model feature columns from bundle, metadata, or fitted pipeline.")

FEATURE_COLUMNS = expected_feature_columns()
print(f"Model expects {len(FEATURE_COLUMNS)} feature columns.")

def load_feature_row(siren, prediction_year=None):
    siren = normalize_siren(siren)
    year_clause = "" if prediction_year is None else f"AND prediction_year = {int(prediction_year)}"
    query = f"""
        SELECT * FROM read_parquet('{FEATURES_GLOB_SQL}', union_by_name=true)
        WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)}
        {year_clause}
        ORDER BY prediction_year DESC LIMIT 1
    """
    con = duckdb.connect()
    try:
        df = con.execute(query).df()
    finally:
        con.close()
    if df.empty:
        years_query = f"""
            SELECT DISTINCT prediction_year FROM read_parquet('{FEATURES_GLOB_SQL}', union_by_name=true)
            WHERE CAST(siren AS VARCHAR) = {sql_literal(siren)} ORDER BY prediction_year
        """
        con = duckdb.connect()
        try:
            years = con.execute(years_query).df()["prediction_year"].dropna().astype(int).tolist()
        finally:
            con.close()
        if years:
            raise ValueError(f"No feature row for SIREN {siren} year {prediction_year}. Available years: {years}")
        raise ValueError(f"No feature row found for SIREN {siren}. Rebuild company_year_features if needed.")
    return df

def align_features(df):
    aligned = df.copy()
    missing = [c for c in FEATURE_COLUMNS if c not in aligned.columns]
    for column in missing:
        aligned[column] = np.nan
    for column in aligned.columns:
        if pd.api.types.is_bool_dtype(aligned[column]):
            aligned[column] = aligned[column].astype(float)
    return aligned[FEATURE_COLUMNS]

def maybe_calibrate(raw_probability):
    if calibrator is None:
        return None
    raw = np.asarray(raw_probability, dtype=float)
    try:
        return calibrator.predict(raw)
    except Exception:
        return calibrator.predict(raw.reshape(-1, 1))

def risk_band(probability):
    if pd.isna(probability):
        return "unknown"
    p = float(probability)
    if p >= 0.50: return "high"
    if p >= 0.30: return "medium"
    if p >= 0.10: return "watch"
    return "low"

# -----------------------------------------------------------------------------
# Confidence score
# -----------------------------------------------------------------------------
# We surface a 0-1 confidence number alongside the probability, combining two
# signals so the UI can distinguish "the model is sure AND the data is rich"
# from "the model is sure but it barely saw anything for this company".
#
#   decisiveness = 2 * |calibrated_prob - 0.5|
#       1.0 when the model gives 0% or 100%, 0.0 when it sits on the fence.
#
#   data_quality = weighted sum of four signals (each 0-1):
#       identity_present  (0.40): did INSEE return a row for this SIREN?
#       financials_present(0.30): do we have at least one filed bilan?
#       recent_filing     (0.20): how stale is the most recent annual filing?
#       cutoff_freshness  (0.10): how recent is the prediction_year vs today?
#
#   confidence = sqrt(decisiveness * data_quality)   # geometric mean -- low if EITHER is low
#
# A row with confidence < 0.3 should be treated as "manual review" rather than
# trusted as-is. Tune the weights below to match what your users actually care
# about; the formula is otherwise stable.

CONFIDENCE_WEIGHTS = {
    "identity_present":   0.40,
    "financials_present": 0.30,
    "recent_filing":      0.20,
    "cutoff_freshness":   0.10,
}

def _compute_data_quality(row):
    """Return (data_quality_score, list_of_reasons, components) for one feature row."""
    reasons = []
    components = {}

    name = row.get("company_name")
    components["identity_present"] = 0.0 if pd.isna(name) or not str(name).strip() else 1.0
    if components["identity_present"] < 1.0:
        reasons.append("no INSEE identity")

    revenue = row.get("latest_revenue")
    components["financials_present"] = 0.0 if pd.isna(revenue) else 1.0
    if components["financials_present"] < 1.0:
        reasons.append("no financials filed")

    years_since = row.get("years_since_last_financial_statement")
    if pd.isna(years_since):
        components["recent_filing"] = 0.0
        if components["financials_present"] >= 1.0:
            reasons.append("filing date missing")
    else:
        ys = float(years_since)
        if   ys <= 2: components["recent_filing"] = 1.0
        elif ys <= 5: components["recent_filing"] = 0.5
        else:
            components["recent_filing"] = 0.0
            reasons.append(f"last filing {ys:.0f}y old")

    pred_year = row.get("prediction_year")
    if pd.isna(pred_year):
        components["cutoff_freshness"] = 0.0
    else:
        current_year = date.today().year
        gap = current_year - int(pred_year)
        if   gap <= 1: components["cutoff_freshness"] = 1.0
        elif gap == 2: components["cutoff_freshness"] = 0.7
        elif gap == 3: components["cutoff_freshness"] = 0.4
        else:
            components["cutoff_freshness"] = 0.2
            reasons.append(f"cutoff {gap}y old")

    score = sum(CONFIDENCE_WEIGHTS[k] * v for k, v in components.items())
    return float(score), reasons, components

def _compute_confidence(probability_for_decision, row):
    if pd.isna(probability_for_decision):
        return float("nan"), float("nan"), float("nan"), "no probability"
    decisiveness = 2.0 * abs(float(probability_for_decision) - 0.5)
    data_quality, reasons, _ = _compute_data_quality(row)
    confidence = float(np.sqrt(decisiveness * data_quality))
    if confidence >= 0.7:
        label = "high confidence"
    elif confidence >= 0.4:
        label = "medium confidence"
    else:
        label = "low confidence"
    if reasons:
        label = f"{label} (" + ", ".join(reasons) + ")"
    return confidence, decisiveness, data_quality, label

def score_siren(siren, prediction_year=None):
    df = load_feature_row(siren, prediction_year)
    X = align_features(df)
    raw = model.predict_proba(X)[:, 1].astype(float)
    calibrated = maybe_calibrate(raw)

    out = pd.DataFrame(index=df.index)
    for column in ["siren", "prediction_year", "prediction_date", "company_name", "activity_code",
                   "legal_category_code", "administrative_status_at_cutoff", "company_age_years",
                   "legal_events_count_12m", "legal_risk_events_count_12m",
                   "days_since_last_legal_event", "annual_accounts_count_24m",
                   "latest_revenue", "latest_net_result", "years_since_last_financial_statement"]:
        if column in df.columns:
            out[column] = df[column].values
    out["probability_raw"] = raw
    if calibrated is not None:
        out["probability_calibrated"] = np.asarray(calibrated, dtype=float)
        score_for_band = out["probability_calibrated"]
    else:
        score_for_band = out["probability_raw"]
    out["risk_band"] = score_for_band.map(risk_band)

    # Confidence — combines model decisiveness with data quality (see helpers above).
    confidences, decisivenesses, qualities, notes = [], [], [], []
    for i in range(len(out)):
        prob = score_for_band.iloc[i]
        c, d, q, n = _compute_confidence(prob, out.iloc[i])
        confidences.append(c)
        decisivenesses.append(d)
        qualities.append(q)
        notes.append(n)
    out["confidence"]              = confidences
    out["confidence_decisiveness"] = decisivenesses
    out["confidence_data_quality"] = qualities
    out["confidence_notes"]        = notes

    out["model_version"] = selected_metadata.get("model_version")
    return out.reset_index(drop=True)

def display_score(result):
    columns = [c for c in [
        "siren", "prediction_year", "company_name", "activity_code",
        "administrative_status_at_cutoff", "company_age_years",
        "legal_risk_events_count_12m", "latest_revenue",
        "probability_raw", "probability_calibrated", "risk_band",
        "confidence", "confidence_notes",
    ] if c in result.columns]
    prob_columns = [c for c in columns if c.startswith("probability") or c == "confidence"]
    formatters = {c: "{:.2%}" for c in prob_columns}
    display(result[columns].style.format(formatters))


## Score one SIREN

Set `SIREN` below. Leave `PREDICTION_YEAR = None` to use the latest cutoff available for this company; set a specific year (e.g. `2024`) to score against that fixed cutoff.

In [ ]:
SIREN = ''               # e.g. '444560502'
PREDICTION_YEAR = None   # None = latest available

if SIREN:
    result = score_siren(SIREN, PREDICTION_YEAR)
    display_score(result)
else:
    print('Set SIREN to a 9-digit value, then run this cell.')

## Batch score

Score multiple SIRENs at once. Output is sorted by calibrated probability descending (or raw if no calibrator) so the riskiest companies are at the top.

In [ ]:
SIRENS = []  # e.g. ['444560502', '552120222', '542065305']
BATCH_PREDICTION_YEAR = None

if SIRENS:
    frames = []
    errors = []
    for value in SIRENS:
        try:
            frames.append(score_siren(value, BATCH_PREDICTION_YEAR))
        except Exception as exc:
            errors.append({'siren': str(value), 'error': str(exc)})
    if frames:
        batch = pd.concat(frames, ignore_index=True)
        sort_col = 'probability_calibrated' if 'probability_calibrated' in batch.columns else 'probability_raw'
        batch = batch.sort_values(sort_col, ascending=False).reset_index(drop=True)
        display_score(batch)
    if errors:
        print('\nSIRENs not scored:')
        display(pd.DataFrame(errors))
else:
    print('Add one or more SIRENs to SIRENS, then run this cell.')

## Diagnostic: what does the data lake know about this SIREN?

**When to use this.** If a score surprises you (low score on a company you know is in trouble, or high score on one that's clearly healthy), check what the data lake actually contains. The most common cause of a surprising score is missing data — a BODACC liquidation announcement that hasn't been ingested, an INPI formality the clean pipeline didn't capture, etc.

**What it shows.**

- The most recent INSEE administrative status (active, cessée, etc.)
- All BODACC legal events for this SIREN, with flags (radiation, liquidation, redressement, ...)
- All INPI formality events
- All annual account filing dates
- All pre-built feature rows (the actual model input across years)

If the BODACC table shows zero rows for a company you *know* has had legal events, your BODACC dataset is incomplete (see `01_data_pipeline.ipynb` to refresh).

In [ ]:
def _resolve(*candidates):
    for c in candidates:
        c = Path(c)
        if c.exists() and any(c.rglob('*.parquet')):
            return (c / '**' / '*.parquet').as_posix().replace("'", "''")
    return None

_IDENT_SQL = _resolve(DATA_LAKE / 'clean' / 'company_identity')
_LEGAL_SQL = _resolve(DATA_LAKE / 'clean' / 'legal_events')
_FORM_SQL  = _resolve(DATA_LAKE / 'clean' / 'formalities_events')
_ACC_SQL   = _resolve(DATA_LAKE / 'clean' / 'annual_accounts')

def diagnose_siren(siren_value):
    siren = normalize_siren(siren_value)
    siren_lit = sql_literal(siren)
    con = duckdb.connect()
    try:
        print(f'=== SIREN {siren} ===')

        if _IDENT_SQL:
            print('\n-- INSEE identity (most recent period first) --')
            df = con.execute(f"""
                SELECT company_name, activity_code, administrative_status,
                       period_start, period_end, closure_date
                FROM read_parquet('{_IDENT_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY COALESCE(period_start, DATE '1900-01-01') DESC LIMIT 5
            """).df()
            display(df)

        if _LEGAL_SQL:
            print('\n-- BODACC events (most recent first) --')
            df = con.execute(f"""
                SELECT event_date, event_category, event_type, is_radiation,
                       flag_liquidation, flag_redressement, flag_sauvegarde, flag_procedure_collective
                FROM read_parquet('{_LEGAL_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY event_date DESC LIMIT 20
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        if _FORM_SQL:
            print('\n-- INPI formalities --')
            df = con.execute(f"""
                SELECT event_date, event_type, event_text
                FROM read_parquet('{_FORM_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY event_date DESC LIMIT 10
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        if _ACC_SQL:
            print('\n-- Annual accounts filings --')
            df = con.execute(f"""
                SELECT filing_date FROM read_parquet('{_ACC_SQL}', union_by_name=true)
                WHERE CAST(siren AS VARCHAR) = {siren_lit}
                ORDER BY filing_date DESC LIMIT 10
            """).df()
            print(f'  ({len(df)} rows)')
            if not df.empty: display(df)

        print('\n-- Pre-built feature rows --')
        df = con.execute(f"""
            SELECT prediction_year, administrative_status_at_cutoff,
                   legal_events_count_12m, legal_risk_events_count_12m,
                   days_since_last_legal_event, annual_accounts_count_24m,
                   latest_revenue, latest_net_result
            FROM read_parquet('{FEATURES_GLOB_SQL}', union_by_name=true)
            WHERE CAST(siren AS VARCHAR) = {siren_lit}
            ORDER BY prediction_year DESC
        """).df()
        print(f'  ({len(df)} rows)')
        if not df.empty: display(df)
    finally:
        con.close()

# Example:
# diagnose_siren('444560502')

## Interpretation guide

**Reading the output.**

- `probability_raw` — internal HGB output. Inflated by class balancing. Use for ranking, not absolute probability.
- `probability_calibrated` — what you should quote to humans. "~6% chance of stopping being active in the next 12 months."
- `risk_band` — coarse bucket derived from the calibrated score (or raw if no calibrator): `low < 10%`, `watch 10-30%`, `medium 30-50%`, `high ≥50%`.

**If a score surprises you:**

1. Run `diagnose_siren(SIREN)` on the company.
2. Check the `prediction_year` of the feature row used. If it's 2024 and the company collapsed in 2026, the score is correct *for what the model saw* — but it didn't see 2025/2026 events. Refresh features in `01_data_pipeline.ipynb`.
3. Check the BODACC and INPI sections. If they're empty for a company you *know* had events, your data lake is missing them — rerun the relevant download in `01_data_pipeline.ipynb` (BODACC with `--mode current` is the most common gap).
4. Check `administrative_status_at_cutoff`. INSEE lags BODACC by months on closures, so a company can be `'A'` in INSEE while actually in liquidation per BODACC.

**What this notebook does NOT do (intentionally):**

- No "rules layer" that bumps the score based on post-cutoff events. The right fix for stale scores is to refresh the data lake and rebuild features, not to bolt rules onto an out-of-date model output. Rules-on-top-of-stale-data is duct tape; you maintain two systems with overlapping logic.
- No retraining trigger. Train in `02_training.ipynb`.

**Reading the confidence column:**

| confidence | meaning |
|---|---|
| ≥ 0.7 | High confidence: the model is decisive AND the data behind it is rich. Trust the probability. |
| 0.4–0.7 | Medium confidence: one of the two is weak (the model is on the fence OR the data is thin). Treat as a heads-up, not a final answer. |
| < 0.4 | Low confidence: probably both. Manual review recommended — don't surface the probability to end users without a caveat. |

`confidence_notes` always spells out *why* confidence is low when it is (e.g. *"low confidence (no financials filed)"*).